# Exploring the MATH dataset

## Load the dataset

In [1]:
from datasets import load_dataset, concatenate_datasets

subjects = [
    "algebra",
    "counting_and_probability",
    "geometry",
    "intermediate_algebra",
    "number_theory",
    "prealgebra",
    "precalculus",
]

all_train = []
all_test = []

for subject in subjects:
    ds = load_dataset("EleutherAI/hendrycks_math", subject)
    all_train.append(ds["train"])
    all_test.append(ds["test"])

train_ds = concatenate_datasets(all_train)
test_ds = concatenate_datasets(all_test)

print(train_ds)
print(test_ds)
example = train_ds[0]
print(example)

Dataset({
    features: ['problem', 'level', 'type', 'solution'],
    num_rows: 7500
})
Dataset({
    features: ['problem', 'level', 'type', 'solution'],
    num_rows: 5000
})
{'problem': 'Let \\[f(x) = \\left\\{\n\\begin{array}{cl} ax+3, &\\text{ if }x>2, \\\\\nx-5 &\\text{ if } -2 \\le x \\le 2, \\\\\n2x-b &\\text{ if } x <-2.\n\\end{array}\n\\right.\\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).', 'level': 'Level 5', 'type': 'Algebra', 'solution': 'For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \\Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be equal when $x=-2$. Substituting, we get $-2-5=2(-2)-b$, which implies $b=3$. So $a+b=-3+3=\\boxed{0}$.'}


## Pretty Print an Example

In [2]:
from IPython.display import display, Markdown
from math_verify import parse, verify


def view_problem(example, show_solution=True):
    gold = parse(example["solution"])[0]
    md = f"""
## {example["type"]} — {example["level"]}

### Problem

{example["problem"]}
"""
    if show_solution:
        md += f"""

### Solution

{example["solution"]}

### Parsed Solution

{gold}

"""
    display(Markdown(md))


view_problem(example)


## Algebra — Level 5

### Problem

Let \[f(x) = \left\{
\begin{array}{cl} ax+3, &\text{ if }x>2, \\
x-5 &\text{ if } -2 \le x \le 2, \\
2x-b &\text{ if } x <-2.
\end{array}
\right.\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).


### Solution

For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be equal when $x=-2$. Substituting, we get $-2-5=2(-2)-b$, which implies $b=3$. So $a+b=-3+3=\boxed{0}$.

### Parsed Solution

0



## Verify Model's Output

In [3]:
def verify_math_llm_answer(problem: dict, llm_answer: str) -> bool:
    if "solution" not in problem:
        raise KeyError("problem must contain a 'solution' field")

    gold = parse(problem["solution"])
    pred = parse(llm_answer)

    return verify(gold, pred)

model_output = r"We solve the equation and get \boxed{0}."
verify_math_llm_answer(example, model_output)

True

In [4]:
model_output = r"We solve the equation and get \boxed{0}."
verify_math_llm_answer(example, model_output)

True